# 第 15 课｜位置表示、LayerNorm 与残差连接

**让模型区分位置，学会逐 token 归一化，并把子层连接成可继续堆叠的结构。**

前一课已经解决了数据、变长批次和验证指标。这一课回到模型内部，准备组装 Transformer。

本课主线：

`Token Embedding → 加位置表示 → LayerNorm → FFN → 残差相加`

这条主线**还没有注意力**。它能区分“相同字符在不同位置”，但不能读取其他位置的内容；不要把它称为完整 Transformer。

本课交付一个可复用的 `PreNormFFN` 和一个两位置分类探针。CPU 即可，不下载数据或模型。Notebook 与配套 `.py` 都独立完整，不需要同目录下的其他课程文件。

**学习方式：** 第 1–4 节学习位置；第 5–8 节手算 LayerNorm 和残差；第 9–12 节做对照实验、保存和自动检查；最后独立完成三组练习。

## 0. 环境与运行约定

沿用已经能运行第 14 课的 PyTorch 环境，无需重新安装。尚未安装时可在单独的 Notebook 单元使用 `%pip install torch`，安装后重启内核；显卡版按官方安装选择器选择 [7]，不要为这一课盲目更换驱动。

默认 CPU，已有可用 CUDA 环境时把 `choose_device("cpu")` 改成 `choose_device("auto")`。这里的微型实验不需要 GPU。

下面打印实际环境。输出写入**当前工作目录**下的 `lesson_15_outputs`，同名 `metrics.json` 和 `position_probe.pt` 会覆盖；没有要修改的外部语料文件。检查失败应修正后重跑，不要跳过失败单元。

只有最后的实验保存步骤才写入这两个输出；定义函数、导入配套脚本不会运行训练。完整自动检查会在临时目录测试保存恢复。

In [1]:
from __future__ import annotations
import argparse
import json
import math
from pathlib import Path
import tempfile
import torch
from torch import nn
from torch.nn import functional as F

def choose_device(requested: str = "cpu") -> torch.device:
    if requested not in {"cpu", "cuda", "auto"}:
        raise ValueError("device 必须是 cpu、cuda 或 auto")
    if requested == "auto":
        requested = "cuda" if torch.cuda.is_available() else "cpu"
    if requested == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA 不可用，请先用 --device cpu，或检查驱动与 PyTorch")
    return torch.device(requested)


def positive_int(value: int, name: str) -> None:
    if type(value) is not int or value <= 0:
        raise ValueError(f"{name} 必须是正整数")


def validate_hidden(x: torch.Tensor, d_model: int) -> None:
    """本课约定：[B,T,D]，每一维非空，浮点数；不接受空批次。"""
    if not isinstance(x, torch.Tensor) or x.ndim != 3:
        raise ValueError("隐藏状态必须是 [B,T,D] 三维张量")
    if not x.is_floating_point() or any(s <= 0 for s in x.shape):
        raise ValueError("隐藏状态必须是各维非空的浮点张量")
    if x.shape[-1] != d_model:
        raise ValueError(f"最后一维应为 {d_model}，实际为 {x.shape[-1]}")


def validate_position_range(start_pos: int, length: int, max_len: int) -> None:
    if type(start_pos) is not int or start_pos < 0:
        raise ValueError("start_pos 必须是非负整数")
    if start_pos + length > max_len:
        raise ValueError(f"位置范围超出 max_len={max_len}；不会自动截断或取模")

# 只为这类小型 CPU 实验减少线程调度开销，不是大模型通用性能设置。
torch.set_num_threads(1)
torch.manual_seed(15)
DEVICE = choose_device("cpu")
OUTPUT_DIR = Path("lesson_15_outputs")
print("PyTorch：", torch.__version__)
print("CUDA 可用：", torch.cuda.is_available(), "本课设备：", DEVICE)
print("输出目录：", OUTPUT_DIR.resolve())

PyTorch： 2.10.0+cpu
CUDA 可用： False 本课设备： cpu
输出目录： /mnt/data/_lesson15_work/notebook_run/lesson_15_outputs


## 1. 先分清：token 编号和位置编号

`token_id` 回答“这是哪个符号”；`position_id` 回答“它在序列哪个位置”。它们是两套不同的编号。

例如 `我爱我` 的字符编号可以是 `[0,1,0]`，位置编号则是 `[0,1,2]`。位置 0 与位置 2 的字符相同，普通 Embedding 会查到同一行。`nn.Embedding` 是可训练的查找表 [2]。

**先预测：** 下面 `token_vectors[0,0]` 与 `token_vectors[0,2]` 是否相同？改变它们在列表中的位置，会自动让 Embedding 知道位置吗？

In [2]:
token_embedding = nn.Embedding(2, 4).to(DEVICE)
with torch.no_grad():
    token_embedding.weight.copy_(torch.tensor([[1., 0., 0., 1.], [0., 2., 1., 0.]], device=DEVICE))
ids = torch.tensor([[0, 1, 0]], dtype=torch.long, device=DEVICE)
token_vectors = token_embedding(ids)
print("token IDs：", ids.tolist())
print("position IDs：", list(range(ids.shape[1])))
print("Embedding 形状：", tuple(token_vectors.shape))
print("相同 token 的向量相同：", torch.equal(token_vectors[0, 0], token_vectors[0, 2]))

token IDs： [[0, 1, 0]]
position IDs： [0, 1, 2]
Embedding 形状： (1, 3, 4)
相同 token 的向量相同： True


## 2. 可学习绝对位置表示：再准备一张表

本课使用加法：

\[
H_{b,t,:}=E[\mathrm{token\_id}_{b,t}]+P[t]
\]

`E` 的形状是 `[词表大小,D]`；`P` 的形状是 `[max_len,D]`。输入 `H` 保持 `[B,T,D]`，**不是拼接成 `[B,T,2D]`**。原始 Transformer 讨论了固定和可学习的位置表示，以及与 token 向量相加的方式 [1]；本课不额外乘原论文的 Embedding 缩放因子，以便单独观察位置加法。

| 张量 | 形状 | 含义 |
|---|---|---|
| `positions` | `[T]` | 本次使用的位置编号 |
| `table(positions)` | `[T,D]` | 位置向量 |
| `.unsqueeze(0)` 后 | `[1,T,D]` | 补出 batch 轴 |
| 与 `x` 相加 | `[B,T,D]` | 相同位置表沿 batch 轴广播 |

注意前几课的广播：这里的广播是有意的。我们并没有让不同 batch 相互交换信息。

In [3]:
class LearnedPositionEmbedding(nn.Module):
    """给 [B,T,D] 加上 [1,T,D] 的可学习绝对位置向量。"""

    def __init__(self, max_len: int, d_model: int) -> None:
        super().__init__()
        positive_int(max_len, "max_len")
        positive_int(d_model, "d_model")
        self.max_len = max_len
        self.d_model = d_model
        self.table = nn.Embedding(max_len, d_model)

    def forward(self, x: torch.Tensor, start_pos: int = 0) -> torch.Tensor:
        validate_hidden(x, self.d_model)
        validate_position_range(start_pos, x.shape[1], self.max_len)
        if x.device != self.table.weight.device or x.dtype != self.table.weight.dtype:
            raise ValueError("输入与位置表的 device/dtype 必须一致，请使用 module.to(...)")
        positions = torch.arange(start_pos, start_pos + x.shape[1], device=x.device)
        position_vectors = self.table(positions)       # [T,D]
        return x + position_vectors.unsqueeze(0)      # [B,T,D] + [1,T,D]

learned_positions = LearnedPositionEmbedding(max_len=6, d_model=4).to(DEVICE)
with torch.no_grad():
    # 只为看清加法而填入确定值；正常模型的位置参数可以随机初始化后训练。
    learned_positions.table.weight.copy_(torch.arange(24, device=DEVICE).reshape(6, 4).float() / 10)
with_positions = learned_positions(token_vectors)
print("加位置后的形状：", tuple(with_positions.shape))
print("位置 0 的‘我’：", with_positions[0, 0].detach().cpu().tolist())
print("位置 2 的‘我’：", with_positions[0, 2].detach().cpu().tolist())
print("现在是否仍相同：", torch.equal(with_positions[0, 0], with_positions[0, 2]))
print("位置表参数量：", sum(p.numel() for p in learned_positions.parameters()))

加位置后的形状： (1, 3, 4)
位置 0 的‘我’： [1.0, 0.10000000149011612, 0.20000000298023224, 1.2999999523162842]
位置 2 的‘我’： [1.7999999523162842, 0.8999999761581421, 1.0, 2.0999999046325684]
现在是否仍相同： False
位置表参数量： 24


### 2.1 位置偏移与最大长度：不要悄悄取模

`start_pos=2` 表示本次片段的位置从 2 开始，例如长度 3 的片段使用 `[2,3,4]`。这个参数只演示编号偏移，**没有实现 KV cache 或分块注意力**。

本课约定 `0 <= start_pos`，且 `start_pos + T <= max_len`；超界报错，不自动截断、不取模。位置表里“存在某一行”也不等于“训练中用过这行”。把位置表预建得很长，不能证明模型会处理长上下文。

参数表通过反向传播更新。下面的损失取求和，只用于追踪查表梯度：一个位置在两个 batch 中各出现一次，对应参数行的梯度就是 2。

In [4]:
print("偏移后的首位置向量：", learned_positions(torch.zeros(1, 3, 4, device=DEVICE), start_pos=2)[0, 0].detach().cpu().tolist())
try:
    learned_positions(torch.zeros(1, 5, 4, device=DEVICE), start_pos=2)
except ValueError as error:
    print("预期的越界错误：", error)

grad_positions = LearnedPositionEmbedding(5, 4).to(DEVICE)
zeros = torch.zeros(2, 3, 4, device=DEVICE, requires_grad=True)
grad_positions(zeros).sum().backward()
print("位置表梯度（前三行被使用、后两行未使用）：")
print(grad_positions.table.weight.grad.cpu())
print("输入梯度是否全为 1：", torch.equal(zeros.grad, torch.ones_like(zeros)))

偏移后的首位置向量： [0.800000011920929, 0.8999999761581421, 1.0, 1.100000023841858]
预期的越界错误： 位置范围超出 max_len=6；不会自动截断或取模
位置表梯度（前三行被使用、后两行未使用）：
tensor([[2., 2., 2., 2.],
        [2., 2., 2., 2.],
        [2., 2., 2., 2.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]])
输入梯度是否全为 1： True


## 3. 固定正弦位置编码：一种不训练参数的位置表示

原始 Transformer 给出了不同频率的 sin/cos 形式 [1]：

\[
PE_{p,2i}=\sin\left(p/10000^{2i/D}\right),\qquad
PE_{p,2i+1}=\cos\left(p/10000^{2i/D}\right)
\]

现在先理解：位置 `p` 输入一组确定的函数，得到长度为 `D` 的向量；偶数列用 sin，奇数列用 cos。**这不是对 token 编号计算三角函数。**

`0::2` 表示从第 0 列开始，每隔两列取一列；`1::2` 表示第 1、3、5……列。若 `D` 是奇数，cos 列比 sin 列少一列，代码需要截取匹配的角度。

此公式可以在更多位置计算，但**能算出数值不等于模型能可靠泛化到未训练长度**。下面为了明确资源与范围，仍预先建立有限长度的表。

In [5]:
def sinusoidal_table(max_len: int, d_model: int) -> torch.Tensor:
    """固定的 sin/cos 位置表；支持奇数 D，构造结果为 CPU float32。"""
    positive_int(max_len, "max_len")
    positive_int(d_model, "d_model")
    positions = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)  # [L,1]
    even_dims = torch.arange(0, d_model, 2, dtype=torch.float32)          # [ceil(D/2)]
    inverse_scales = torch.exp(-math.log(10000.0) * even_dims / d_model)
    angles = positions * inverse_scales
    table = torch.zeros(max_len, d_model)
    table[:, 0::2] = torch.sin(angles)
    # D 是奇数时，cos 列比 sin 列少一列。
    n_cos = table[:, 1::2].shape[1]
    table[:, 1::2] = torch.cos(angles[:, :n_cos])
    return table

fixed_table = sinusoidal_table(4, 4)
print("4 个位置、4 个维度：")
print(fixed_table)
print("位置 0 的向量：", fixed_table[0].tolist())
print("奇数维度也支持：", tuple(sinusoidal_table(3, 5).shape))

4 个位置、4 个维度：
tensor([[ 0.0000,  1.0000,  0.0000,  1.0000],
        [ 0.8415,  0.5403,  0.0100,  0.9999],
        [ 0.9093, -0.4161,  0.0200,  0.9998],
        [ 0.1411, -0.9900,  0.0300,  0.9996]])
位置 0 的向量： [0.0, 1.0, 0.0, 1.0]
奇数维度也支持： (3, 5)


### 3.1 `register_buffer`：要保存和移动，但不交给优化器训练

固定位置表不是 `nn.Parameter`。把它注册为 **buffer** 后，模块会管理它：随 `.to(device/dtype)` 移动或转换，默认进入 `state_dict()`，但不出现在 `parameters()` 中 [4]。

注册 buffer 不代表任意张量永远不能有梯度；是这里构造的固定表本身 `requires_grad=False`，且没有当作训练参数。输入仍然可以有梯度。

区别是：**可学习表是一组要更新的参数；固定表是模型需要携带的状态。**

In [6]:
class SinusoidalPositionEncoding(nn.Module):
    """固定位置表注册为 buffer：随 .to() 移动、保存，但不是可训练参数。"""

    def __init__(self, max_len: int, d_model: int) -> None:
        super().__init__()
        self.max_len = max_len
        self.d_model = d_model
        self.register_buffer("table", sinusoidal_table(max_len, d_model))

    def forward(self, x: torch.Tensor, start_pos: int = 0) -> torch.Tensor:
        validate_hidden(x, self.d_model)
        validate_position_range(start_pos, x.shape[1], self.max_len)
        if x.device != self.table.device or x.dtype != self.table.dtype:
            raise ValueError("输入与位置表的 device/dtype 必须一致，请使用 module.to(...)")
        selected = self.table[start_pos:start_pos + x.shape[1]]
        return x + selected.unsqueeze(0)

fixed_positions = SinusoidalPositionEncoding(6, 4).to(DEVICE)
print("输出形状：", tuple(fixed_positions(token_vectors).shape))
print("可学习参数数目：", sum(p.numel() for p in fixed_positions.parameters()))
print("buffer 名称：", list(dict(fixed_positions.named_buffers())))
print("state_dict 键：", list(fixed_positions.state_dict()))
print("buffer requires_grad：", fixed_positions.table.requires_grad)

输出形状： (1, 3, 4)
可学习参数数目： 0
buffer 名称： ['table']
state_dict 键： ['table']
buffer requires_grad： False


## 4. 一个更准确的顺序实验：无位置、无 mask 时的注意力

回顾第 10 课，这里只用 `Q=K=V=X`，做一个**不带任何 mask** 的内容注意力实验：

\[
f(X)=\operatorname{softmax}(XX^\top/\sqrt D)X
\]

重排输入行，再计算输出，等于把原输出按相同顺序重排。这叫**置换等变**，不是“输出完全不变”：对于行重排 `P`，有 `f(PX)=P f(X)`。可以从点积矩阵同时重排行、列，以及 softmax 按行运算推导出来；下面用数值验证。

加入位置后，只重排 token 内容、仍按位置 0/1/2 加位置向量，结果通常不再是简单重排。

**适用边界：** 本实验没有因果掩码。因果掩码本身就与顺序相关，不能把这个性质直接套用到带因果掩码的模型，或断言所有不使用显式位置表的模型都无法感知任何顺序。

In [7]:
def content_only_attention(x: torch.Tensor) -> torch.Tensor:
    """置换实验专用：Q=K=V=x；无位置、无 mask；不是本课训练模型的一部分。"""
    if not isinstance(x, torch.Tensor) or x.ndim != 3:
        raise ValueError("x 应为 [B,T,D]")
    validate_hidden(x, x.shape[-1])
    scores = x @ x.transpose(-2, -1) / math.sqrt(x.shape[-1])
    return scores.softmax(dim=-1) @ x

sequence = torch.tensor([[[1., 0., 0., 1.], [0., 2., 1., 0.], [1., 1., -1., 0.]]], device=DEVICE)
permutation = torch.tensor([2, 0, 1], device=DEVICE)
without_position_error = (content_only_attention(sequence[:, permutation]) - content_only_attention(sequence)[:, permutation]).abs().max()
position_module = SinusoidalPositionEncoding(3, 4).to(DEVICE)
with_position_error = (content_only_attention(position_module(sequence[:, permutation])) - content_only_attention(position_module(sequence))[:, permutation]).abs().max()
print("无位置：两种路径的最大差异（接近 0）：", without_position_error.item())
print("固定位置编号后：两种路径的最大差异：", with_position_error.item())

无位置：两种路径的最大差异（接近 0）： 1.1920928955078125e-07
固定位置编号后：两种路径的最大差异： 0.9028334617614746


## 5. LayerNorm：只在当前 token 的特征维度上统计

本课始终使用 `nn.LayerNorm(D)`，输入为 `[B,T,D]`。这意味着：对每个 `(b,t)` 的 **D 个特征**分别计算均值和方差，不跨 token，也不跨 batch [3]。

对一个向量 `x=[x1,...,xD]`：

\[
\mu=\frac1D\sum_i x_i,\quad
\sigma^2=\frac1D\sum_i(x_i-\mu)^2
\]
\[
\hat{x}_i=\frac{x_i-\mu}{\sqrt{\sigma^2+\epsilon}},\qquad
 y_i=\gamma_i\hat{x}_i+\beta_i
\]

`gamma` 和 `beta` 都是长度 D 的可学习向量，分别对应 PyTorch 的 `.weight` 和 `.bias`，默认初始化为 1 和 0 [3]。它们不是均值或方差的缓存。

**手算：** `[1,2,3,4]` 的均值为 2.5，平方偏差和为 5，方差为 `5/4=1.25`。这里分母是 **D，不是 D−1**。使用 `torch.var` 核对时要写 `unbiased=False`（等价于 `correction=0`）。

In [8]:
one_token = torch.tensor([1., 2., 3., 4.], dtype=torch.float64, device=DEVICE)
mu = one_token.mean()
variance = ((one_token - mu) ** 2).mean()
normalized = (one_token - mu) / torch.sqrt(variance + 1e-5)
print("均值：", mu.item(), "方差：", variance.item())
print("归一化结果：", normalized.cpu().tolist())
print("结果均值：", normalized.mean().item())
print("结果方差：", normalized.var(unbiased=False).item())
print("理论方差 var/(var+eps)：", (variance / (variance + 1e-5)).item())

均值： 2.5 方差： 1.25
归一化结果： [-1.3416354199689269, -0.447211806656309, 0.447211806656309, 1.3416354199689269]
结果均值： 0.0
结果方差： 0.9999920000639994
理论方差 var/(var+eps)： 0.9999920000639995


### 5.1 为什么“方差永远精确等于 1”是错误的？

加入 `eps` 后，仿射变换之前的方差是：

\[
\operatorname{Var}(\hat{x})=\frac{\sigma^2}{\sigma^2+\epsilon}
\]

常量向量的方差是 0，归一化结果是 0，而不是某个方差为 1 的向量；再乘 gamma、加 beta 后，结果就是 beta。可学习仿射变换也会改变最终输出的均值和方差。

`eps` 在**平方根里面**。不是先取标准差再加 eps。

下面手写版本只支持 float32/float64，用来理解计算和梯度；不声称它具备 PyTorch 内核的全部低精度稳定性与性能。

In [9]:
def manual_layer_norm(
    x: torch.Tensor, weight: torch.Tensor, bias: torch.Tensor, eps: float = 1e-5
) -> torch.Tensor:
    """最后一维 LayerNorm 的教学实现；限定 float32/float64，不实现混合精度内核。"""
    if not isinstance(x, torch.Tensor) or x.ndim < 1 or x.numel() == 0:
        raise ValueError("x 必须是非空、至少一维的张量")
    if x.dtype not in {torch.float32, torch.float64}:
        raise ValueError("手写版仅支持 float32/float64；低精度训练使用 nn.LayerNorm")
    if isinstance(eps, bool) or not isinstance(eps, (int, float)) or not math.isfinite(eps) or eps <= 0:
        raise ValueError("eps 必须是有限正数")
    for name, parameter in [("weight", weight), ("bias", bias)]:
        if not isinstance(parameter, torch.Tensor) or parameter.shape != (x.shape[-1],):
            raise ValueError(f"{name} 必须是与最后一维对应的 [D] 张量")
        if parameter.dtype != x.dtype or parameter.device != x.device:
            raise ValueError(f"{name} 的 dtype/device 必须与 x 一致")
    mean = x.mean(dim=-1, keepdim=True)
    variance = ((x - mean) ** 2).mean(dim=-1, keepdim=True)  # 分母 D，不是 D-1
    normalized = (x - mean) / torch.sqrt(variance + eps)
    return normalized * weight + bias

layer_input = torch.tensor([[[1., 2., 3., 4.], [10., 20., 30., 40.]],
                            [[-3., 0., 2., 7.], [5., 5., 5., 5.]]], dtype=torch.float64, device=DEVICE)
ln = nn.LayerNorm(4).to(device=DEVICE, dtype=torch.float64)
manual_output = manual_layer_norm(layer_input, ln.weight, ln.bias)
official_output = ln(layer_input)
print("每个 token 的均值：", layer_input.mean(-1).cpu())
print("手写 / 官方最大差异：", (manual_output - official_output).abs().max().item())
print("常量 token 输出：", official_output[1, 1].detach().cpu().tolist())
print("LayerNorm 参数量：", sum(p.numel() for p in ln.parameters()))

每个 token 的均值： tensor([[ 2.5000, 25.0000],
        [ 1.5000,  5.0000]], dtype=torch.float64)
手写 / 官方最大差异： 2.220446049250313e-16
常量 token 输出： [0.0, 0.0, 0.0, 0.0]
LayerNorm 参数量： 8


### 5.2 检查“没有跨 token 统计”与“能够反向传播”

把另一个 token 改得很大，本 token 的 `LayerNorm(D)` 结果应保持不变。

再比较手写实现与官方实现对输入、gamma、beta 的梯度。损失使用不同特征的加权和，而不是归一化输出简单求和：简单求和可能因零均值约束出现接近零的输入梯度，容易误判程序断了计算图。

`LayerNorm(D)` 在 train/eval 下都使用当前输入统计 [3]；**这不是所有网络层都共有的行为**，例如后面引入的 Dropout 会受模式影响。

In [10]:
altered = layer_input.clone()
altered[0, 1] = torch.tensor([100., -30., 7., 200.], dtype=torch.float64, device=DEVICE)
print("改其他 token 后，本 token 输出不变：", torch.allclose(ln(altered)[0, 0], ln(layer_input)[0, 0]))
train_output = ln.train()(layer_input).detach()
eval_output = ln.eval()(layer_input).detach()
print("仅 LayerNorm 的 train/eval 输出一致：", torch.allclose(train_output, eval_output))

x_manual = layer_input.clone().requires_grad_()
x_official = layer_input.clone().requires_grad_()
gamma = torch.ones(4, dtype=torch.float64, device=DEVICE, requires_grad=True)
beta = torch.zeros(4, dtype=torch.float64, device=DEVICE, requires_grad=True)
reference_ln = nn.LayerNorm(4).to(device=DEVICE, dtype=torch.float64)
coefficient = torch.arange(16, dtype=torch.float64, device=DEVICE).reshape(2, 2, 4) / 10
(manual_layer_norm(x_manual, gamma, beta) * coefficient).sum().backward()
(reference_ln(x_official) * coefficient).sum().backward()
print("输入梯度最大差异：", (x_manual.grad - x_official.grad).abs().max().item())
print("gamma 梯度最大差异：", (gamma.grad - reference_ln.weight.grad).abs().max().item())
print("beta 梯度最大差异：", (beta.grad - reference_ln.bias.grad).abs().max().item())

改其他 token 后，本 token 输出不变： True
仅 LayerNorm 的 train/eval 输出一致： True
输入梯度最大差异： 5.684341886080802e-14
gamma 梯度最大差异： 2.220446049250313e-16
beta 梯度最大差异： 0.0


## 6. 残差连接：保留输入，再加一个可学习的修正

\[
 y=x+F(x)
\]

`F(x)` 是分支输出。残差学习把层组织为在输入之上学习修正，而不是完全替换输入 [5]。如果分支是零，输出仍然是 x。

本课要求两侧同形状、同 dtype、同 device，不允许广播。它与“加位置向量时有意广播”不同：这里希望 FFN 确实为每个位置产生完整的 `[B,T,D]` 修正。

**不要写 `x.detach() + F(x)`**：虽然前向数值可能相同，但捷径的输入梯度被切断了。也避免 `x += ...` 这类原地修改带来的额外自动求导问题。

在可微处，残差的导数中有一条恒等路径：`dy/dx = I + J_F`。这不保证所有深度与学习率下都不会梯度消失或爆炸；分支的导数仍然会参与计算，甚至可能抵消恒等项。

In [11]:
def residual_add(x: torch.Tensor, delta: torch.Tensor) -> torch.Tensor:
    """本课残差连接严格同形状，不允许广播掩盖维度错误；不原地修改、不 detach。"""
    if not isinstance(x, torch.Tensor) or not isinstance(delta, torch.Tensor):
        raise ValueError("残差两侧都必须是张量")
    if x.shape != delta.shape:
        raise ValueError(f"残差两侧形状必须相同：{tuple(x.shape)} != {tuple(delta.shape)}")
    if x.device != delta.device or x.dtype != delta.dtype:
        raise ValueError("残差两侧 dtype/device 必须相同")
    return x + delta

residual_input = torch.tensor([[[1., -2., 3., 4.]]], device=DEVICE, requires_grad=True)
zero_branch = torch.zeros_like(residual_input)
residual_output = residual_add(residual_input, zero_branch)
residual_output.sum().backward()
print("零分支时保留输入：", torch.equal(residual_output.detach(), residual_input.detach()))
print("输入梯度：", residual_input.grad.cpu())
try:
    residual_add(torch.zeros(2, 3, 4, device=DEVICE), torch.zeros(1, 3, 4, device=DEVICE))
except ValueError as error:
    print("预期的形状错误：", error)

零分支时保留输入： True
输入梯度： tensor([[[1., 1., 1., 1.]]])
预期的形状错误： 残差两侧形状必须相同：(2, 3, 4) != (1, 3, 4)


## 7. Pre-LN 与 Post-LN：加法和归一化放在哪里？

两种结构不要混写 [1][6]：

\[
\text{Pre-LN}:\quad y=x+F(\operatorname{LN}(x))
\]
\[
\text{Post-LN}:\quad y=\operatorname{LN}(x+F(x))
\]

“Pre/Post”指相对于分支变换和残差合并的位置。原始 Transformer 使用 Post-LN；本课程后续组装选择 Pre-LN，但不把某一种结构说成所有任务上的绝对最优。

做一个不用训练的对照：把 `F` 设置为全零。Pre-LN 得到 `x`，Post-LN 得到 `LN(x)`。它们显然不是同一个函数。

In [12]:
ordering_x = torch.tensor([[[1., 2., 3., 4.]]], device=DEVICE)
ordering_ln = nn.LayerNorm(4).to(DEVICE)
pre_ln_zero = ordering_x + torch.zeros_like(ordering_ln(ordering_x))
post_ln_zero = ordering_ln(ordering_x + torch.zeros_like(ordering_x))
print("Pre-LN 零分支：", pre_ln_zero.detach().cpu().tolist())
print("Post-LN 零分支：", post_ln_zero.detach().cpu().tolist())
print("二者是否相同：", torch.allclose(pre_ln_zero, post_ln_zero))

Pre-LN 零分支： [[[1.0, 2.0, 3.0, 4.0]]]
Post-LN 零分支： [[[-1.3416354656219482, -0.4472118318080902, 0.4472118318080902, 1.3416354656219482]]]
二者是否相同： False


## 8. 组装 PreNormFFN：这是子层，不是整层 Transformer

沿用第 12 课的 MLP，使用：

\[
 y=x+W_2\operatorname{GELU}(W_1\operatorname{LN}(x)+b_1)+b_2
\]

上式按列向量记法理解；PyTorch `Linear` 接收最后一维特征，代码不用手动转置权重。

| 步骤 | 形状 |
|---|---|
| 输入 / LayerNorm | `[B,T,D]` |
| 第一个 Linear / GELU | `[B,T,F]` |
| 第二个 Linear | `[B,T,D]` |
| 残差相加 | `[B,T,D]` |

取 `D=4,F=8`：LayerNorm 8 个参数，第一层 40 个，第二层 36 个，总计 **84**。残差加法没有额外参数。

还缺少的部分是自注意力子层：FFN 只处理每个位置自身的特征，不把序列长度 T 当作全连接特征维度。本课不引入 Dropout，避免它干扰结构对照。

In [13]:
class PreNormFFN(nn.Module):
    """y = x + Linear2(GELU(Linear1(LayerNorm(x))))。不是完整 Transformer。"""

    def __init__(self, d_model: int, hidden_dim: int) -> None:
        super().__init__()
        positive_int(d_model, "d_model")
        positive_int(hidden_dim, "hidden_dim")
        self.d_model = d_model
        self.norm = nn.LayerNorm(d_model)
        self.fc1 = nn.Linear(d_model, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        validate_hidden(x, self.d_model)
        delta = self.fc2(F.gelu(self.fc1(self.norm(x))))
        return residual_add(x, delta)

torch.manual_seed(15)
ffn_block = PreNormFFN(d_model=4, hidden_dim=8).to(DEVICE)
ffn_input = torch.randn(2, 3, 4, device=DEVICE, requires_grad=True)
ffn_output = ffn_block(ffn_input)
ffn_loss = ffn_output.square().mean()
ffn_loss.backward()
print("输入/输出形状：", tuple(ffn_input.shape), tuple(ffn_output.shape))
print("参数量：", sum(p.numel() for p in ffn_block.parameters()))
for name, parameter in ffn_block.named_parameters():
    print(name, tuple(parameter.shape), "梯度绝对值之和：", parameter.grad.abs().sum().item())

changed_ffn_input = ffn_input.detach().clone()
changed_ffn_input[:, 0] += torch.tensor([1., -3., 2., 5.], device=DEVICE)
with torch.no_grad():
    untouched_difference = (ffn_block(changed_ffn_input)[:, 1:] - ffn_block(ffn_input.detach())[:, 1:]).abs().max()
print("改第 0 个位置，其他位置输出最大变化：", untouched_difference.item())

输入/输出形状： (2, 3, 4) (2, 3, 4)
参数量： 84
norm.weight (4,) 梯度绝对值之和： 0.0915127694606781
norm.bias (4,) 梯度绝对值之和： 0.06112762168049812
fc1.weight (8, 4) 梯度绝对值之和： 1.707784652709961
fc1.bias (8,) 梯度绝对值之和： 0.39874354004859924
fc2.weight (4, 8) 梯度绝对值之和： 1.9850062131881714
fc2.bias (4,) 梯度绝对值之和： 0.8375260233879089
改第 0 个位置，其他位置输出最大变化： 0.0


### 8.1 归零残差分支，验证数值与梯度两条路径

只在这个测试副本中，把第二层 Linear 的权重和偏置设为零，让分支输出恒为零。此时 `block(x)` 应等于 `x`，对 `block(x).sum()` 求导得到全 1 的输入梯度。

这个检查不是建议把整个模型所有参数归零初始化。当最后一层归零时，第一层等更早分支参数在这一轮还可能得到零梯度；分支更新后的行为需要继续观察。前一单元才是非零初始化下的完整梯度检查。

In [14]:
zero_block = PreNormFFN(4, 8).to(DEVICE)
with torch.no_grad():
    zero_block.fc2.weight.zero_()
    zero_block.fc2.bias.zero_()
zero_input = torch.randn(1, 2, 4, device=DEVICE, requires_grad=True)
zero_output = zero_block(zero_input)
zero_output.sum().backward()
print("与输入一致：", torch.equal(zero_output.detach(), zero_input.detach()))
print("梯度全 1：", torch.equal(zero_input.grad, torch.ones_like(zero_input)))
print("第一层本轮梯度之和：", zero_block.fc1.weight.grad.abs().sum().item())

与输入一致： True
梯度全 1： True
第一层本轮梯度之和： 0.0


## 9. 对照实验：相同 token 能否按位置区分？

这次故意不用上一课的文本训练流程，而构造一个**机制探针**：

```text
输入 token IDs：[0, 0]
分类标签：       [0, 1]
```

第一个位置要输出类别 0，第二个位置要输出类别 1。**这些是位置分类标签，不是右移得到的下一个 token 标签。**只有这一条两位置训练输入，没有验证集或测试集。目标是排除复杂语料，让信息是否可用变得清楚；不能据此评估语言能力、记忆力或泛化能力。

对照结构：

- 无位置：`Embedding → PreNormFFN → LayerNorm → Linear(2类)`。
- 有位置：`Embedding + 可学习位置向量 → 相同的后续结构`。

两模型共有部分从**相同初始权重**开始，优化器和步数相同。有位置的模型多了一张 2×8 的位置表。无位置模型用同一个逐位置函数处理两个相同输入，必须给出相同 logits；即使调整参数，也无法让两个位置的确定性分类同时正确。

In [15]:
class PositionProbe(nn.Module):
    """只做两个位置的分类探针，没有跨 token 交互，没有自注意力。"""

    def __init__(self, use_positions: bool, d_model: int = 8, max_len: int = 2) -> None:
        super().__init__()
        if type(use_positions) is not bool:
            raise ValueError("use_positions 必须是 bool")
        positive_int(d_model, "d_model")
        positive_int(max_len, "max_len")
        self.use_positions = use_positions
        self.d_model = d_model
        self.max_len = max_len
        self.token_embedding = nn.Embedding(2, d_model)
        self.positions = LearnedPositionEmbedding(max_len, d_model) if use_positions else None
        self.block = PreNormFFN(d_model, 2 * d_model)
        self.final_norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, 2)

    def forward(self, ids: torch.Tensor) -> torch.Tensor:
        if not isinstance(ids, torch.Tensor) or ids.ndim != 2 or ids.dtype != torch.long:
            raise ValueError("ids 必须是 torch.long 的 [B,T] 张量")
        if any(s <= 0 for s in ids.shape) or ids.shape[1] > self.max_len:
            raise ValueError("批次和序列不能为空，T 不能超过 max_len")
        if bool(((ids < 0) | (ids >= 2)).any()):
            raise ValueError("本探针只支持 token ID 0 和 1")
        hidden = self.token_embedding(ids)
        if self.positions is not None:
            hidden = self.positions(hidden)
        hidden = self.block(hidden)
        return self.head(self.final_norm(hidden))


def probe_metrics(model: PositionProbe, ids: torch.Tensor, targets: torch.Tensor) -> dict:
    """只汇报探针训练输入上的拟合情况，恢复原 train/eval 模式。"""
    was_training = model.training
    model.eval()
    try:
        with torch.no_grad():
            logits = model(ids)
            return {
                "loss": float(F.cross_entropy(logits.reshape(-1, 2), targets.reshape(-1)).item()),
                "accuracy": float((logits.argmax(dim=-1) == targets).float().mean().item()),
                "probabilities": logits.softmax(dim=-1)[0].cpu().tolist(),
                "predictions": logits.argmax(dim=-1)[0].cpu().tolist(),
            }
    finally:
        model.train(was_training)


def run_position_experiment(device: torch.device, steps: int = 180, seed: int = 15) -> tuple[dict, dict]:
    """相同 token，位置 0 标为类别 0，位置 1 标为类别 1；不是 next-token 标签。"""
    positive_int(steps, "steps")
    torch.manual_seed(seed)
    # 先在 CPU 构造，然后移动，便于理解初始化与设备转换的区别。
    models = {
        "without_positions": PositionProbe(False).to(device),
        "with_positions": PositionProbe(True).to(device),
    }
    # 配对对照：两模型共有部分使用完全相同的初始参数，仅有位置表的差别。
    common = dict(models["without_positions"].named_parameters())
    with torch.no_grad():
        for name, parameter in models["with_positions"].named_parameters():
            if name in common:
                parameter.copy_(common[name])
    ids = torch.tensor([[0, 0]], dtype=torch.long, device=device)
    targets = torch.tensor([[0, 1]], dtype=torch.long, device=device)
    metrics = {
        "task": "two_position_training_probe_not_a_language_benchmark",
        "device": str(device), "torch_version": str(torch.__version__),
        "steps": steps, "seed": seed, "learning_rate": 0.03,
        "input_ids": [[0, 0]], "targets": [[0, 1]],
        "without_positions_loss_lower_bound": math.log(2),
        "scope": "只有一个两位置训练样本，无验证集或测试集，不衡量语言理解和长上下文能力。",
        "models": {},
    }
    for name, model in models.items():
        before = probe_metrics(model, ids, targets)
        optimizer = torch.optim.Adam(model.parameters(), lr=0.03)
        history = []
        model.train()
        for step in range(steps):
            optimizer.zero_grad(set_to_none=True)
            logits = model(ids)
            loss = F.cross_entropy(logits.reshape(-1, 2), targets.reshape(-1))
            if not bool(torch.isfinite(loss)):
                raise RuntimeError(f"{name} 在 step={step} 出现非有限损失")
            loss.backward()
            optimizer.step()
            history.append(float(loss.item()))
        after = probe_metrics(model, ids, targets)
        metrics["models"][name] = {
            "parameters": sum(p.numel() for p in model.parameters()),
            "before": before, "after": after, "training_loss_before_each_update": history,
        }
    # 改第 0 个 token，固定第 1 个 token 和位置；逐位置模型的第 1 个输出不应改变。
    with torch.no_grad():
        model = models["with_positions"].eval()
        alternate = torch.tensor([[1, 0]], dtype=torch.long, device=device)
        difference = (model(ids)[:, 1] - model(alternate)[:, 1]).abs().max().item()
    metrics["other_token_change_last_logits_max_diff"] = float(difference)
    return models, metrics

models, experiment_metrics = run_position_experiment(DEVICE, steps=180, seed=15)
for model_name, result in experiment_metrics["models"].items():
    after = result["after"]
    print(model_name)
    print("  参数量：", result["parameters"])
    print("  初始 loss：", result["before"]["loss"])
    print("  训练后 loss：", after["loss"])
    print("  训练位置准确率：", after["accuracy"])
    print("  预测：", after["predictions"])
    print("  两个位置的概率：", after["probabilities"])

without_positions
  参数量： 346
  初始 loss： 0.9096946120262146
  训练后 loss： 0.6931471824645996
  训练位置准确率： 0.5
  预测： [0, 0]
  两个位置的概率： [[0.5000081062316895, 0.4999919533729553], [0.5000081062316895, 0.4999919533729553]]
with_positions
  参数量： 362
  初始 loss： 0.5086551308631897
  训练后 loss： 1.3768569260719232e-05
  训练位置准确率： 1.0
  预测： [0, 1]
  两个位置的概率： [[0.9999904632568359, 9.58704731601756e-06], [1.8052525774692185e-05, 0.9999819993972778]]


### 9.1 为什么无位置模型的损失下界是 log(2)？

两个位置输出相同分布 `[p,1-p]`，但标签分别是 0、1：

\[
L=-\frac{\log p+\log(1-p)}2
=-\frac12\log[p(1-p)]\ge\log2
\]

因为 `p(1-p)` 最大为 1/4，在 `p=0.5` 时取到。对两个位置作相同的 argmax 预测，只能正确一个，准确率为 50%。这是**本探针模型和数据下的可推导限制**，不是所有无位置编码网络的统一上限。

有位置模型能够输出不同的分布，但训练是否收敛仍取决于配置。看本机实测数值，不要把预期现象替代实际结果。

In [16]:
plain_result = experiment_metrics["models"]["without_positions"]["after"]
position_result = experiment_metrics["models"]["with_positions"]["after"]
print("无位置的理论下界：", math.log(2))
print("本次训练结果减下界：", plain_result["loss"] - math.log(2))
print("有位置的实测准确率：", position_result["accuracy"])
print("说明：这里没有独立测试集，数字只用于验证信息可区分性。")

无位置的理论下界： 0.6931471805599453
本次训练结果减下界： 1.904654323148236e-09
有位置的实测准确率： 1.0
说明：这里没有独立测试集，数字只用于验证信息可区分性。


## 10. 一个重要反例：知道位置，仍然不等于读取上下文

取训练后的有位置模型，对比 `[0,0]` 与 `[1,0]`。第一位置内容改变，但第二位置的 token 和位置编号都没变。

由于模型所有子层仍然逐位置运算，第二位置输出不会变化。这个结果可以精确定位当前能力的边界：**它知道本位置的编号，却没有读取前一个 token。**

下一课增加多头因果自注意力后，再做同样的“改前文看后文 / 改未来看过去”实验，分别检查上下文交互和因果性。

In [17]:
probe = models["with_positions"].eval()
with torch.no_grad():
    original_logits = probe(torch.tensor([[0, 0]], device=DEVICE))
    changed_logits = probe(torch.tensor([[1, 0]], device=DEVICE))
print("第二位置 logits 最大差异：", (original_logits[:, 1] - changed_logits[:, 1]).abs().max().item())
print("实验汇总中的差异：", experiment_metrics["other_token_change_last_logits_max_diff"])

第二位置 logits 最大差异： 0.0
实验汇总中的差异： 0.0


## 11. 接回上一课的 padding：不要混淆三种用途

| 设置 | 解决的问题 | 本课是否实现 |
|---|---|---|
| 位置编号 | 为 token 提供位置线索 | 是，绝对位置，统一 offset |
| `ignore_index=-100` | 不把补齐标签计入损失 | 第 14 课已实现，本探针无 padding |
| attention 的 padding/causal mask | 哪些位置允许读取哪些位置 | 本主模型尚无 attention，下一课组装 |

即使 PAD 的 token Embedding 是零，**加上位置向量后也通常不再是零**。LayerNorm、带偏置的 Linear 也不能代替显式掩码。屏蔽损失不等于屏蔽注意力读取。

本课位置表直接使用从 0 开始的物理下标，适合先理解固定长度、右侧 padding。左侧 padding 或打包序列往往需要按样本构造不同的 `position_ids`；本实现没有自动识别这些语义，不要直接搬去生产系统。

In [18]:
pad_vectors = torch.zeros(1, 3, 4, device=DEVICE)
with torch.no_grad():
    pad_after_positions = position_module(pad_vectors)
print("零 token 向量加位置后：")
print(pad_after_positions.cpu())
print("是否仍然全零：", torch.equal(pad_after_positions, torch.zeros_like(pad_after_positions)))

零 token 向量加位置后：
tensor([[[ 0.0000,  1.0000,  0.0000,  1.0000],
         [ 0.8415,  0.5403,  0.0100,  0.9999],
         [ 0.9093, -0.4161,  0.0200,  0.9998]]])
是否仍然全零： False


## 12. 自动检查、保存和重载

下面的固定检查覆盖位置边界、奇数 D、buffer、LayerNorm 数值和梯度、残差直通路径、严格形状、逐位置性，以及固定/可学习位置的保存恢复。

**测试与训练对照分开：** 固定测试不要求某次优化一定达到指定准确率。它们检查函数应满足的约定；训练曲线属于实验观测。

这一单元较长，可以先运行，再按检查组回读。它和配套脚本的 `run_checks()` 是同一份代码。

In [19]:
def run_checks(device: torch.device | str = "cpu") -> int:
    """性质、边界和梯度检查；不以探针训练准确率作为固定单元测试。"""
    device = torch.device(device)
    torch.manual_seed(150)
    count = 0

    def check(condition, label):
        nonlocal count
        if not bool(condition):
            raise AssertionError(label)
        count += 1

    def close(actual, expected, label, atol=1e-6, rtol=1e-5):
        nonlocal count
        try:
            torch.testing.assert_close(actual, expected, atol=atol, rtol=rtol)
        except AssertionError as error:
            raise AssertionError(label) from error
        count += 1

    def raises(call, label, error_type=ValueError):
        nonlocal count
        try:
            call()
        except error_type:
            count += 1
        else:
            raise AssertionError(label + "：没有抛出预期异常")

    # A. 可学习位置表：形状、offset、广播、边界、梯度。
    pos = LearnedPositionEmbedding(5, 4).to(device)
    with torch.no_grad():
        pos.table.weight.copy_(torch.arange(20, device=device).reshape(5, 4).float())
    x = torch.zeros(2, 3, 4, device=device)
    y = pos(x)
    check(y.shape == x.shape, "位置加法保持形状")
    close(y[0], pos.table.weight[:3], "位置逐行选择")
    close(y[1], y[0], "同一位置广播到不同 batch")
    check(not torch.equal(y[:, 0], y[:, 1]), "不同位置可有不同向量")
    close(pos(x, 2)[0], pos.table.weight[2:5], "start_pos 偏移")
    close(x, torch.zeros_like(x), "不原地改写输入")
    check(sum(p.numel() for p in pos.parameters()) == 20, "可学习位置参数量")
    check("table.weight" in pos.state_dict(), "位置参数进入 state_dict")
    pos(x).sum().backward()
    close(pos.table.weight.grad[:3], torch.full((3, 4), 2.0, device=device), "位置梯度按 batch 累积")
    close(pos.table.weight.grad[3:], torch.zeros(2, 4, device=device), "未使用位置没有查表梯度")
    for call, label in [
        (lambda: pos(x, -1), "拒绝负 offset"),
        (lambda: pos(x, True), "拒绝布尔 offset"),
        (lambda: pos(x, 3), "拒绝超出位置表"),
        (lambda: pos(torch.zeros(2, 3, 5, device=device)), "拒绝错误 D"),
        (lambda: pos(torch.zeros(3, 4, device=device)), "拒绝缺少 B 轴"),
        (lambda: pos(torch.zeros(2, 0, 4, device=device)), "拒绝空 T"),
        (lambda: pos(x.long()), "拒绝整型隐藏状态"),
        (lambda: pos(x.double()), "拒绝 dtype 不一致"),
        (lambda: LearnedPositionEmbedding(0, 4), "拒绝零 max_len"),
        (lambda: LearnedPositionEmbedding(4, True), "拒绝布尔 D"),
    ]:
        raises(call, label)

    # B. 固定位置表：buffer、奇数 D、位置偏移和输入梯度。
    pe = SinusoidalPositionEncoding(5, 4).to(device)
    close(pe.table[0], torch.tensor([0., 1., 0., 1.], device=device), "sin/cos 位置零")
    check(pe.table.shape == (5, 4), "固定位置表形状")
    check(sum(p.numel() for p in pe.parameters()) == 0, "固定位置表无可学习参数")
    check("table" in dict(pe.named_buffers()), "位置表为 buffer")
    check("table" in pe.state_dict(), "持久 buffer 保存到 state_dict")
    close(pe(x, 2)[0], pe.table[2:5], "固定表 offset")
    check(sinusoidal_table(3, 5).shape == (3, 5), "支持奇数 D")
    close(sinusoidal_table(2, 1)[:, 0], torch.sin(torch.arange(2).float()), "支持 D=1")
    check(torch.isfinite(sinusoidal_table(8, 7)).all(), "固定表数值有限")
    input_grad = torch.zeros_like(x, requires_grad=True)
    pe(input_grad).sum().backward()
    close(input_grad.grad, torch.ones_like(x), "固定位置加法保留输入梯度")
    check(pe.table.grad is None and not pe.table.requires_grad, "buffer 不自动求梯度")
    raises(lambda: pe(x, 3), "固定位置超界")
    raises(lambda: pe(x, -1), "固定位置负 offset")
    double_pe = SinusoidalPositionEncoding(5, 4).to(device=device, dtype=torch.float64)
    check(double_pe.table.dtype == torch.float64, ".to(dtype) 同时转换 buffer")

    # C. 手写 LayerNorm：值、轴、参数、输入和参数梯度。
    base = torch.tensor([[[1., 2., 3., 4.], [2., 5., 8., 11.]],
                         [[-3., 0., 2., 7.], [4., 4., 4., 4.]]], device=device, dtype=torch.float64)
    w = torch.tensor([1.0, 0.5, -1.0, 2.0], device=device, dtype=torch.float64)
    b = torch.tensor([0.3, -0.2, 0.7, 1.1], device=device, dtype=torch.float64)
    official = nn.LayerNorm(4).to(device=device, dtype=torch.float64)
    with torch.no_grad():
        official.weight.copy_(w)
        official.bias.copy_(b)
    close(manual_layer_norm(base, w, b), official(base), "手写与官方一致", 1e-10, 1e-10)
    normalized = manual_layer_norm(base, torch.ones_like(w), torch.zeros_like(b))
    close(normalized.mean(-1), torch.zeros(2, 2, device=device, dtype=torch.float64), "逐 token 均值为零", 1e-10)
    variance = base.var(-1, unbiased=False)
    close(normalized.var(-1, unbiased=False), variance / (variance + 1e-5), "eps 下方差不必精确为一", 1e-10)
    close(manual_layer_norm(base, w, b)[1, 1], b, "常量向量输出 beta", 1e-10)
    check(torch.isfinite(normalized).all(), "常量向量不会除零")
    changed = base.clone()
    changed[1, 0] = torch.tensor([100., -30., 8., 99.], device=device)
    close(official(changed)[0], official(base)[0], "LN 不跨 batch 混合统计")
    close(official(changed)[1, 1], official(base)[1, 1], "LN(D) 不跨 token 混合统计")
    close(official(base).detach(), official.eval()(base).detach(), "LN train/eval 不改统计规则")
    check(sum(p.numel() for p in official.parameters()) == 8, "LN(D) 的参数量为 2D")
    x1 = base.clone().requires_grad_()
    x2 = base.clone().requires_grad_()
    w1, b1 = w.clone().requires_grad_(), b.clone().requires_grad_()
    factor = torch.arange(16, device=device, dtype=torch.float64).reshape(2, 2, 4) / 10
    (manual_layer_norm(x1, w1, b1) * factor).sum().backward()
    (official(x2) * factor).sum().backward()
    close(x1.grad, x2.grad, "LN 输入梯度一致", 1e-9, 1e-8)
    close(w1.grad, official.weight.grad, "LN gamma 梯度一致", 1e-10, 1e-9)
    close(b1.grad, official.bias.grad, "LN beta 梯度一致", 1e-10, 1e-9)
    for call, label in [
        (lambda: manual_layer_norm(base, w, b, 0), "拒绝零 eps"),
        (lambda: manual_layer_norm(base, w, b, float('nan')), "拒绝 NaN eps"),
        (lambda: manual_layer_norm(base, w[:3], b), "拒绝错误 gamma 形状"),
        (lambda: manual_layer_norm(base, w, b.float()), "拒绝不同 dtype"),
        (lambda: manual_layer_norm(base.long(), w, b), "拒绝整数 LN"),
    ]:
        raises(call, label)

    # D. 残差与 Pre-LN：严格形状、恒等路径、参数梯度、逐位置性。
    rx = torch.randn(2, 3, 4, device=device, requires_grad=True)
    close(residual_add(rx, torch.zeros_like(rx)), rx, "零分支保留原输入")
    delta = torch.randn_like(rx, requires_grad=True)
    residual_add(rx, delta).sum().backward()
    close(rx.grad, torch.ones_like(rx), "残差输入直通梯度")
    close(delta.grad, torch.ones_like(delta), "残差分支有梯度")
    raises(lambda: residual_add(rx, torch.zeros(1, 3, 4, device=device)), "拒绝可广播但不同形状的残差")
    raises(lambda: residual_add(rx, rx.double()), "拒绝残差 dtype 不同")
    block = PreNormFFN(4, 8).to(device)
    bx = torch.randn(2, 3, 4, device=device, requires_grad=True)
    by = block(bx)
    check(by.shape == bx.shape, "PreNormFFN 保持 [B,T,D]")
    check(sum(p.numel() for p in block.parameters()) == 84, "4->8->4 残差 FFN 参数量 84")
    by.square().mean().backward()
    check(bx.grad is not None and torch.isfinite(bx.grad).all(), "FFN 输入梯度有限")
    check(all(p.grad is not None and torch.isfinite(p.grad).all() for p in block.parameters()), "FFN 全部参数连接到损失")
    check(all(float(p.grad.abs().sum()) > 0 for p in block.parameters()), "本测试中全部参数梯度非零")
    altered = bx.detach().clone()
    altered[:, 0] += torch.tensor([2., -3., 1., 7.], device=device)
    close(block(altered)[:, 1:], block(bx.detach())[:, 1:], "逐位置 FFN 不会混合其他位置")
    with torch.no_grad():
        block.fc2.weight.zero_()
        block.fc2.bias.zero_()
    zx = torch.randn(1, 2, 4, device=device, requires_grad=True)
    close(block(zx), zx, "最后线性层归零时为恒等映射")
    block(zx).sum().backward()
    close(zx.grad, torch.ones_like(zx), "零分支下恒等路径梯度为一")
    check(not torch.allclose(zx.detach(), block.norm(zx).detach()), "Pre-LN 与 Post-LN 零分支行为不同")
    raises(lambda: block(torch.zeros(1, 2, 5, device=device)), "拒绝 FFN 错误 D")

    # E. 无掩码的内容注意力：置换等变；固定位置编号改变了这个对照。
    seq = torch.tensor([[[1., 0., 0., 1.], [0., 2., 1., 0.], [1., 1., -1., 0.]]], device=device)
    perm = torch.tensor([2, 0, 1], device=device)
    close(content_only_attention(seq[:, perm]), content_only_attention(seq)[:, perm], "无 mask/位置的注意力置换等变")
    sp = SinusoidalPositionEncoding(3, 4).to(device)
    difference = (content_only_attention(sp(seq[:, perm])) - content_only_attention(sp(seq))[:, perm]).abs().max()
    check(difference > 1e-3, "保留位置编号时内容重排产生不同结果")

    # F. 探针能力边界、保存恢复；不运行训练。
    plain, positioned = PositionProbe(False).to(device), PositionProbe(True).to(device)
    ids = torch.tensor([[0, 0]], device=device)
    check(positioned(ids).shape == (1, 2, 2), "探针输出 [B,T,C]")
    close(plain(ids)[:, 0], plain(ids)[:, 1], "无位置模型相同 token 必须同输出")
    alternate = torch.tensor([[1, 0]], device=device)
    close(positioned(ids)[:, 1], positioned(alternate)[:, 1], "加入位置并不产生上下文交互")
    raises(lambda: positioned(ids.float()), "探针拒绝浮点编号")
    raises(lambda: positioned(torch.tensor([[2, 0]], device=device)), "探针拒绝越界编号")
    with tempfile.TemporaryDirectory() as tmp:
        file = Path(tmp) / "position_buffers.pt"
        torch.save({"learned": pos.state_dict(), "fixed": pe.state_dict()}, file)
        state = torch.load(file, map_location=device, weights_only=True)
        restored_pos = LearnedPositionEmbedding(5, 4).to(device)
        restored_fixed = SinusoidalPositionEncoding(5, 4).to(device)
        restored_pos.load_state_dict(state["learned"])
        restored_fixed.load_state_dict(state["fixed"])
        close(restored_pos(x), pos(x), "可学习位置表保存恢复")
        close(restored_fixed(x), pe(x), "固定 buffer 保存恢复")
    print(f"通过固定检查：{count} 项")
    return count

CHECK_COUNT = run_checks(DEVICE)
experiment_metrics["fixed_checks"] = CHECK_COUNT

通过固定检查：75 项


### 12.1 保存实验

保存两模型的权重、构造配置和 JSON 指标。`save_experiment()` 会重新构造模型并加载本程序刚生成的检查点，再对比 logits，而不只检查“文件存在”。固定位置 buffer 的保存恢复已经在上面的临时目录检查中验证。

权重统一保存到 CPU，便于之后在其他设备上加载。只加载可信来源的检查点；本例使用 `weights_only=True`，不存整个 Python 模型对象。文件写入策略是覆盖输出目录中的同名生成文件，不修改课程源文件。

In [20]:
def save_experiment(models: dict, metrics: dict, output_dir: Path) -> tuple[Path, Path]:
    """只写生成文件；保存 CPU 权重与构造信息，不保存任意模型对象。"""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = output_dir / "position_probe.pt"
    metrics_path = output_dir / "metrics.json"
    checkpoint = {"lesson": 15, "models": {}}
    for name, model in models.items():
        checkpoint["models"][name] = {
            "config": {"use_positions": model.use_positions,
                       "d_model": model.d_model, "max_len": model.max_len},
            "state_dict": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
        }
    torch.save(checkpoint, checkpoint_path)
    # 重载并验证预测；只加载本程序刚刚生成、已知来源的文件。
    loaded = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    reference_ids = torch.tensor([[0, 0]], dtype=torch.long)
    for name, entry in loaded["models"].items():
        restored = PositionProbe(**entry["config"]).eval()
        restored.load_state_dict(entry["state_dict"])
        original = models[name]
        original_device = next(original.parameters()).device
        with torch.no_grad():
            actual = restored(reference_ids)
            expected = original(reference_ids.to(original_device)).cpu()
        torch.testing.assert_close(actual, expected, atol=2e-5, rtol=2e-5)
    metrics["checkpoint_reload_matches"] = True
    metrics_path.write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")
    return metrics_path, checkpoint_path

metrics_path, checkpoint_path = save_experiment(models, experiment_metrics, OUTPUT_DIR)
print("指标：", metrics_path.resolve())
print("权重：", checkpoint_path.resolve())
print("重载后 logits 一致：", experiment_metrics["checkpoint_reload_matches"])

指标： /mnt/data/_lesson15_work/notebook_run/lesson_15_outputs/metrics.json
权重： /mnt/data/_lesson15_work/notebook_run/lesson_15_outputs/position_probe.pt
重载后 logits 一致： True


## 13. 独立练习：先写，再打开检查开关

下面的 `RUN_EXERCISE_*` 默认 `False`，保证阅读讲义时可以继续执行；**它们不是已经完成的答案**。完成函数后改为 `True`。练习故意使用与正文不同的输入，防止只记住某个输出。

### 练习 A：位置相加

实现 `add_positions_ex(x, table, start_pos=0)`：

`x` 是 `[B,T,D]`，`table` 是 `[max_len,D]`，均为非空、同设备同 dtype 的浮点张量。截取正确的 T 行并相加，输出 `[B,T,D]`；禁止改写输入；负偏移、超界、错误 D 要报 `ValueError`。不要调用正文的位置类。

思考：位置向量在哪里增加 batch 轴？重复两个 batch 会不会需要两份独立位置表？

In [21]:
def check_exercise_a(fn):
    x = torch.zeros(2, 3, 4, requires_grad=True)
    table = torch.arange(24).reshape(6, 4).float().requires_grad_()
    before = x.detach().clone()
    result = fn(x, table, 2)
    torch.testing.assert_close(result, table[2:5].unsqueeze(0).expand(2, -1, -1))
    torch.testing.assert_close(x.detach(), before)
    result.sum().backward()
    torch.testing.assert_close(x.grad, torch.ones_like(x))
    expected_grad = torch.zeros_like(table)
    expected_grad[2:5] = 2
    torch.testing.assert_close(table.grad, expected_grad)
    for call in [lambda: fn(x, table, -1), lambda: fn(x, table, 4),
                 lambda: fn(x, torch.zeros(6, 5), 0)]:
        try:
            call()
        except ValueError:
            pass
        else:
            raise AssertionError("非法位置/形状没有被拒绝")
    print("练习 A：数值、边界、非原地修改和梯度检查通过。")


def add_positions_ex(x, table, start_pos=0):
    raise NotImplementedError("请实现位置截取、形状检查和加法")

RUN_EXERCISE_A = False
if RUN_EXERCISE_A:
    check_exercise_a(add_positions_ex)
else:
    print("练习 A 尚未验收：完成函数后将开关改为 True。")

练习 A 尚未验收：完成函数后将开关改为 True。


### 练习 B：手写最后一维 LayerNorm

实现 `layer_norm_ex(x, gamma, beta, eps=1e-5)`，只用张量的减、乘、均值、平方根等基本操作，不能调用正文实现、`F.layer_norm` 或 `nn.LayerNorm`。

测试输入已经是合法 float64，重点是数学：在最后一维计算均值；方差分母使用 D；eps 在平方根内；保留求导图。`gamma`、`beta` 是 `[D]`。

**不要**在中间 `.item()` 后重新构造张量，也不要 `.detach()`。检查不仅比较输出，还比较梯度。

In [22]:
def check_exercise_b(fn):
    x = torch.tensor([[[1., 2., 5., 8.], [4., 4., 4., 4.]]], dtype=torch.float64, requires_grad=True)
    gamma = torch.tensor([1., 0.5, 2., -1.], dtype=torch.float64, requires_grad=True)
    beta = torch.tensor([0.1, -0.3, 0.7, 0.5], dtype=torch.float64, requires_grad=True)
    x2 = x.detach().clone().requires_grad_()
    g2 = gamma.detach().clone().requires_grad_()
    b2 = beta.detach().clone().requires_grad_()
    expected = F.layer_norm(x2, (4,), g2, b2, 1e-5)
    actual = fn(x, gamma, beta, 1e-5)
    torch.testing.assert_close(actual, expected, atol=1e-10, rtol=1e-10)
    coefficients = torch.tensor([1., -2., 0.3, 4.], dtype=torch.float64)
    (actual * coefficients).sum().backward()
    (expected * coefficients).sum().backward()
    for a, b in [(x.grad, x2.grad), (gamma.grad, g2.grad), (beta.grad, b2.grad)]:
        if a is None:
            raise AssertionError("计算图被切断，缺少梯度")
        torch.testing.assert_close(a, b, atol=1e-9, rtol=1e-9)
    print("练习 B：普通/常量向量、仿射参数、输入与参数梯度检查通过。")


def layer_norm_ex(x, gamma, beta, eps=1e-5):
    raise NotImplementedError("请实现最后一维 LayerNorm")

RUN_EXERCISE_B = False
if RUN_EXERCISE_B:
    check_exercise_b(layer_norm_ex)
else:
    print("练习 B 尚未验收：完成函数后将开关改为 True。")

练习 B 尚未验收：完成函数后将开关改为 True。


### 练习 C：实现 PreNormFFNEx

创建 `PreNormFFNEx(nn.Module)`，构造函数接收 `d_model, hidden_dim`，层名为 `norm, fc1, fc2`。`norm` 使用 LayerNorm，`fc1` 扩维、GELU 激活、`fc2` 回到 D，最后返回**原输入**加分支结果。

输入约定 `[B,T,D]`，本练习测试的是合法浮点输入。没有 Dropout，不要加入 attention。测试会归零 `fc2` 并验证恒等路径，也会修改一个位置查看其他位置是否保持不变。

In [23]:
def check_exercise_c(factory):
    torch.manual_seed(151)
    model = factory(4, 8)
    x = torch.randn(2, 3, 4, requires_grad=True)
    expected = x + model.fc2(F.gelu(model.fc1(model.norm(x))))
    actual = model(x)
    torch.testing.assert_close(actual, expected)
    if sum(p.numel() for p in model.parameters()) != 84:
        raise AssertionError("D=4、F=8 时应有 84 个参数")
    actual.square().mean().backward()
    if not all(p.grad is not None and torch.isfinite(p.grad).all() for p in model.parameters()):
        raise AssertionError("有参数没有有限梯度")
    changed = x.detach().clone()
    changed[:, 0] += torch.tensor([2., -3., 4., 1.])
    torch.testing.assert_close(model(changed)[:, 1:], model(x.detach())[:, 1:])
    with torch.no_grad():
        model.fc2.weight.zero_()
        model.fc2.bias.zero_()
    z = torch.randn(1, 2, 4, requires_grad=True)
    result = model(z)
    torch.testing.assert_close(result, z)
    result.sum().backward()
    torch.testing.assert_close(z.grad, torch.ones_like(z))
    print("练习 C：结构、参数量、梯度、逐位置性与恒等路径检查通过。")


class PreNormFFNEx(nn.Module):
    def __init__(self, d_model, hidden_dim):
        super().__init__()
        raise NotImplementedError("请定义 norm、fc1、fc2")

    def forward(self, x):
        raise NotImplementedError("请实现 Pre-LN 前馈残差结构")

RUN_EXERCISE_C = False
if RUN_EXERCISE_C:
    check_exercise_c(PreNormFFNEx)
else:
    print("练习 C 尚未验收：完成类后将开关改为 True。")

练习 C 尚未验收：完成类后将开关改为 True。


---
## 14. 参考答案：先完成独立练习，再继续

参考答案使用新名称，不会覆盖你写的练习函数。每个答案单元包含自己的导入和一个最小自检，可以复制到空白 Notebook 单独运行；后面的综合核对单元使用上方较全面的练习检查器。

### A 的一种实现

In [24]:
import torch


def add_positions_answer(x, table, start_pos=0):
    if not isinstance(x, torch.Tensor) or not isinstance(table, torch.Tensor):
        raise ValueError("x/table 必须是张量")
    if x.ndim != 3 or table.ndim != 2 or any(s <= 0 for s in x.shape) or table.shape[0] <= 0:
        raise ValueError("需要非空 [B,T,D] 和 [L,D]")
    if x.shape[-1] != table.shape[-1] or not x.is_floating_point():
        raise ValueError("D 必须一致，且输入必须浮点")
    if x.dtype != table.dtype or x.device != table.device:
        raise ValueError("dtype/device 不一致")
    if type(start_pos) is not int or start_pos < 0 or start_pos + x.shape[1] > table.shape[0]:
        raise ValueError("位置范围不合法")
    selected = table[start_pos:start_pos + x.shape[1]]
    return x + selected.unsqueeze(0)


_answer_x = torch.zeros(2, 2, 3)
_answer_table = torch.arange(12).reshape(4, 3).float()
torch.testing.assert_close(add_positions_answer(_answer_x, _answer_table, 1),
                           _answer_table[1:3].unsqueeze(0).expand(2, -1, -1))
print("A 参考答案最小自检通过。")

A 参考答案最小自检通过。


### B 的一种实现

这里按练习约定只展示合法输入的计算核心；正文函数另含完整输入验证。每一步都是张量运算，所以自动求导可以沿计算图追踪。

In [25]:
import torch
from torch.nn import functional as F


def layer_norm_answer(x, gamma, beta, eps=1e-5):
    mean = x.mean(dim=-1, keepdim=True)
    variance = ((x - mean) ** 2).mean(dim=-1, keepdim=True)
    normalized = (x - mean) / torch.sqrt(variance + eps)
    return normalized * gamma + beta


_answer_x = torch.tensor([[[1., 3., 6.], [4., 4., 4.]]], dtype=torch.float64)
_answer_gamma = torch.tensor([1., 2., -1.], dtype=torch.float64)
_answer_beta = torch.tensor([0.1, 0.2, 0.3], dtype=torch.float64)
torch.testing.assert_close(layer_norm_answer(_answer_x, _answer_gamma, _answer_beta),
                           F.layer_norm(_answer_x, (3,), _answer_gamma, _answer_beta),
                           atol=1e-10, rtol=1e-10)
print("B 参考答案最小自检通过。")

B 参考答案最小自检通过。


### C 的一种实现

`x` 保留为残差主路；归一化后的值只进入分支。若先令 `x = self.norm(x)` 再返回 `x + branch`，主路已经变成归一化后的输入，结构就改变了。

In [26]:
import torch
from torch import nn
from torch.nn import functional as F


class PreNormFFNAnswer(nn.Module):
    def __init__(self, d_model, hidden_dim):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.fc1 = nn.Linear(d_model, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, d_model)

    def forward(self, x):
        branch = self.fc2(F.gelu(self.fc1(self.norm(x))))
        return x + branch


_answer_model = PreNormFFNAnswer(4, 8)
_answer_x = torch.randn(2, 3, 4)
if _answer_model(_answer_x).shape != _answer_x.shape:
    raise AssertionError("输出形状错误")
if sum(p.numel() for p in _answer_model.parameters()) != 84:
    raise AssertionError("参数量错误")
print("C 参考答案最小自检通过。")

C 参考答案最小自检通过。


### 使用同一套练习检查器核对参考实现

参考答案通过不会自动使你的练习版本通过。请回到练习区，打开相应开关检查自己的实现。

In [27]:
check_exercise_a(add_positions_answer)
check_exercise_b(layer_norm_answer)
check_exercise_c(PreNormFFNAnswer)

练习 A：数值、边界、非原地修改和梯度检查通过。
练习 B：普通/常量向量、仿射参数、输入与参数梯度检查通过。


练习 C：结构、参数量、梯度、逐位置性与恒等路径检查通过。


## 15. 验收标准与常见错误

完成本课时，应该能不看答案解释：位置编号与 token 编号为什么不同；`LayerNorm(D)` 为什么不跨 T 统计；残差主路为什么不能 `.detach()`；Pre-LN 与 Post-LN 为什么不等价；为什么本课模型还不能利用前文。

| 现象 | 优先检查 |
|---|---|
| 位置加法把 D 变成 2D | 是否把相加写成了拼接 |
| 改其他 token 导致本 token 的 LN 结果改变 | 是否把 T 纳入了归一化轴 |
| 手写 LN 与官方不一致 | 方差分母、eps 的位置、gamma/beta、dtype |
| 零分支输出不等于输入 | 是否误把残差输入覆盖成 LN(x) |
| 前向相同但梯度不一致 | 是否 detach、item 后重建、或原地更新 |
| 很长输入超界 | 表容量与 offset，不能用取模“修复” |
| 位置探针 100% 就认为模型理解语序 | 它只拟合了两个人工位置标签，且没有上下文交互 |

**和后续模型的关系：** 本课是 Transformer 组装基础，不意味着所有模型都使用可学习绝对位置或 LayerNorm。RoPE 和 RMSNorm 将在后续现代架构课单独实现和比较；不要把它们与本课的“输入相加”和“减均值归一化”混为同一计算。

### 配套脚本

```bash
python lesson_15_positions_layernorm_and_residuals.py
python lesson_15_positions_layernorm_and_residuals.py --checks-only
python lesson_15_positions_layernorm_and_residuals.py --device auto
python lesson_15_positions_layernorm_and_residuals.py --output-dir my_run_15
```

`--steps` 控制探针更新次数，默认 180；改变后应重新观察结果，不保证任意步数都收敛。固定检查用显式异常或 `torch.testing.assert_close`，不是依赖可被 `-O` 移除的 `assert` 语句。

**下一课：第 16 课——多头因果自注意力与 Transformer Block。** 将位置、注意力、LayerNorm、FFN、残差真正组合起来，检查改前文能否影响后文，以及改未来是否保持前缀不变。

## 16. 原始资料与引用

本课数值例子、位置探针、测试与练习为独立构造的教学实验；论文链接用于核对架构来源，官方文档用于核对 API 语义。文档版本可能随更新跳转，以开头打印的本机版本为准。

[1] Vaswani et al., *Attention Is All You Need*, §3.1、§3.3、§3.5。<https://arxiv.org/html/1706.03762v7>

[2] PyTorch, `nn.Embedding`。<https://docs.pytorch.org/docs/stable/generated/torch.nn.Embedding.html>

[3] PyTorch, `nn.LayerNorm`。<https://docs.pytorch.org/docs/stable/generated/torch.nn.LayerNorm.html>

[4] PyTorch, `nn.Module.register_buffer`、`state_dict`。<https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html>

[5] He et al., *Deep Residual Learning for Image Recognition*。<https://arxiv.org/abs/1512.03385>

[6] Xiong et al., *On Layer Normalization in the Transformer Architecture*。<https://arxiv.org/abs/2002.04745>

[7] PyTorch 官方安装选择器。<https://pytorch.org/get-started/locally/>